# 18 Industry-Style Risk Report

Generate a practical portfolio risk report with composition, Greeks, stress losses, VaR, Expected Shortfall, and quantum-versus-classical errors.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
positions = load_portfolio_config()
valued, totals = value_portfolio(positions)
stress = standard_stress_table(positions, config)
pnl = monte_carlo_portfolio_pnl(positions, n_scenarios=3000, horizon_days=float(config["stress"]["monte_carlo_horizon_days"]), seed=int(config["random_seed"]))
risk = var_expected_shortfall(pnl, config["stress"]["var_levels"])
quantum_df, quantum_totals = value_portfolio_quantum(positions, n_qubits=6, x_width=float(config["quantum"]["x_width"]))
value_error = quantum_totals["total_quantum_value"] - totals["total_theoretical_value"]
contributors = valued.assign(abs_vega=valued["position_Vega"].abs()).sort_values("abs_vega", ascending=False)
loss_block = stress[['scenario', 'pnl']].head(5).to_string(index=False)
vega_block = contributors[['label', 'underlying', 'option_type', 'strike', 'position_Vega']].head(5).to_string(index=False)
report_md = f"""
# Industry-Style Risk Report

Synthetic examples only; not live market data.

## Portfolio Value

- Classical theoretical value: {totals['total_theoretical_value']:.2f}
- Market value from sample inputs: {totals['total_market_value']:.2f}
- Model P&L: {totals['total_pnl']:.2f}
- Quantum reconstructed value: {quantum_totals['total_quantum_value']:.2f}
- Quantum minus classical value error: {value_error:.2f}

## Portfolio Greeks

- Delta: {totals['portfolio_Delta']:.4f}
- Gamma: {totals['portfolio_Gamma']:.6f}
- Vega: {totals['portfolio_Vega']:.2f}
- Theta: {totals['portfolio_Theta']:.2f}
- Rho: {totals['portfolio_Rho']:.2f}

## Tail Risk

- VaR 95: {risk['VaR_95']:.2f}
- ES 95: {risk['ES_95']:.2f}
- VaR 99: {risk['VaR_99']:.2f}
- ES 99: {risk['ES_99']:.2f}

## Largest Loss Scenarios

```text
{loss_block}
```

## Largest Vega Contributors

```text
{vega_block}
```

## Limitations

This report is a research artifact. It does not claim quantum advantage, trading readiness, or live-market calibration.
"""
report_path = REPORTS_DIR / "industry_risk_report.md"
report_path.write_text(report_md, encoding="utf-8")
save_table(valued, "18_risk_report_positions.csv")
save_table(stress, "18_risk_report_stress.csv")
save_output(risk, "18_risk_report_var_es.json")
plt.figure()
plt.bar(contributors["label"], contributors["position_Vega"])
plt.xticks(rotation=45, ha="right")
plt.title("Largest risk contributors by Vega")
plt.ylabel("Position Vega")
save_current_figure("18_largest_vega_contributors.png")
print(f"Report written to {report_path}")
report_md[:1000]
